# R7-H2 — Experimento B2: pesos de clase atenuados

**Objetivo.** Evaluar si reducir la intensidad de la ponderación de clases modifica el Average Precision (AP) de ConvNeXt-Tiny con la imagen completa (representación A).

**Hipótesis (H2).** La ponderación por frecuencia inversa usada en R4–R6 (relación aproximada de 8.20 entre las clases maligna y benigna) modifica el AP del checkpoint seleccionado respecto de una ponderación atenuada (relación aproximada de 2.86). No se presupone el sentido del cambio: podría haber mejora, ninguna diferencia o deterioro.

**Diseño.** Se comparan dos configuraciones con las mismas semillas (42, 43 y 44) y el mismo conjunto de validación. El peso de cada clase es proporcional a `frecuencia ** (-α)` y se normaliza para que el peso medio por muestra sea 1:

| | B0 (basal) | B2 |
|---|---|---|
| Origen | Entrenamientos de R4, reutilizados en R6 (solo lectura) | Tres entrenamientos nuevos, ejecutados en Falcon |
| Exponente α | 1.0 (frecuencia inversa) | 0.5 (ponderación atenuada) |
| Pesos (benigna, maligna) | ≈ 0.5610, 4.5977 | ≈ 0.8315, 2.3805 |

Los pesos de B2 no se escriben a mano: se calculan en la sección 3 a partir de las frecuencias de entrenamiento con `compute_class_weights`. Todo lo demás es idéntico a B0: arquitectura, representación A, particiones, hiperparámetros, data augmentation, 30 épocas máximas, **early stopping activo** (paciencia 5, `min_delta` 0.001), checkpoint por máximo `val_roc_auc` y umbral 0.5. B2 se entrena desde los pesos preentrenados originales, no desde los checkpoints de B0 ni de B1.

**Efecto indirecto sobre el early stopping.** El early stopping se basa en la pérdida ponderada de validation, que depende de los pesos de clase. Cambiar los pesos puede por tanto modificar también las épocas de parada. Ese efecto forma parte de la intervención y no se puede separar de ella. Por la misma razón, `train_loss` y `val_loss` (ponderadas) **no son comparables entre B0 y B2**; para comparar pérdidas entre configuraciones se usa `val_unweighted_loss`.

**Cómo usar este notebook.** Las secciones 1–3 leen B0 y preparan B2; la sección 3 puede lanzar los entrenamientos solo si se activa `RUN_TRAINING` (desactivado por defecto). Las secciones 4–5 cargan las salidas de B2 ya completadas y las comparan con B0, y pueden repetirse sin volver a entrenar. No se usa `test`.

## 1. Configuración

In [ ]:
from pathlib import Path
import json
import shlex
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score


def locate_project_root() -> Path:
    current_dir = Path.cwd().resolve()

    for candidate in [current_dir, *current_dir.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate

    raise FileNotFoundError("No se pudo localizar la raíz del proyecto.")


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classification.utils import compute_class_weights  # mismo cálculo de pesos que usa run_experiment

ARCH = "convnext_tiny"
CONDITION = "A"
SEEDS = [42, 43, 44]
EPOCHS_MAX = 30
THRESHOLD = 0.5
ALPHA = 0.5  # exponente de la ponderación de B2 (B0: 1.0)
EXPERIMENT_NAME = "r7_class_weights"  # etiqueta que run_experiment registra en el run_config de B2

CLASSIFICATION_DIR = PROJECT_ROOT / "results" / "classification"
B0_DIR = CLASSIFICATION_DIR / "r4_benchmark" / ARCH / CONDITION  # solo lectura
B2_DIR = CLASSIFICATION_DIR / "r7_optimizacion" / "class_weights"
ANALYSIS_DIR = B2_DIR / "analysis"
SPLIT_MANIFEST = PROJECT_ROOT / "configs" / "splits" / "mmotu_experimental_split_standard.csv"

HISTORY_KEYS = [
    "train_loss", "train_acc", "val_loss", "val_unweighted_loss", "val_acc", "val_roc_auc",
    "val_average_precision", "val_f1_malignant", "val_recall_malignant", "val_specificity", "val_balanced_accuracy",
]
# Nombres reales en los archivos: `val_loss` es la pérdida ponderada de validation y la sensibilidad es `recall_malignant`
SAVED_KEYS = {"roc_auc": "roc_auc", "average_precision": "average_precision", "sensitivity": "recall_malignant", "specificity": "specificity"}

COLORS = {"B0": "#0072B2", "B2": "#009E73"}
LABELS = {"B0": "B0 (α = 1.0)", "B2": f"B2 (α = {ALPHA})"}

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})


def b0_dir(seed):
    return B0_DIR / f"seed_{seed}"


def b2_dir(seed):
    return B2_DIR / f"seed_{seed}"


def require(condition, message):
    if not condition:
        raise ValueError(message)


def compute_metrics(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_pred = (np.asarray(y_prob) >= THRESHOLD).astype(int)
    return {
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "average_precision": float(average_precision_score(y_true, y_prob)),
        "sensitivity": float((y_pred[y_true == 1] == 1).mean()),
        "specificity": float((y_pred[y_true == 0] == 0).mean()),
    }


manifest = pd.read_csv(SPLIT_MANIFEST)
validation_manifest = manifest[manifest["experimental_split"] == "validation"].set_index("image_id")


def load_run(directory: Path, seed: int) -> dict:
    # Carga una corrida y comprueba que checkpoint, historial, predicciones y métricas sean coherentes entre sí
    with open(directory / "run_config.json", encoding="utf-8") as f:
        config = json.load(f)
    with open(directory / "training_history.json", encoding="utf-8") as f:
        raw_history = json.load(f)
    with open(directory / "validation_metrics.json", encoding="utf-8") as f:
        saved_metrics = json.load(f)
    predictions = pd.read_csv(directory / "val_predictions_best_auc.csv")

    history = pd.DataFrame({key: raw_history[key] for key in HISTORY_KEYS})
    history.index = pd.RangeIndex(1, len(history) + 1, name="epoch")

    require(config["seed"] == seed, f"{directory}: la semilla del run_config no coincide con la carpeta.")
    require(len(history) == config["epochs_run"], f"{directory}: el historial no tiene {config['epochs_run']} épocas.")
    # idxmax devuelve la primera época en caso de empate, igual que el criterio de checkpoint
    require(int(history["val_roc_auc"].idxmax()) == config["epoch_best"], f"{directory}: el checkpoint no es el máximo de val_roc_auc.")
    require(
        predictions["epoch"].eq(config["epoch_best"]).all() and len(predictions) == len(validation_manifest),
        f"{directory}: las predicciones no corresponden al checkpoint o no cubren las 200 imágenes de validation.",
    )
    labels_in_manifest = validation_manifest["binary_label"].reindex(predictions["image_id"])
    require(
        labels_in_manifest.notna().all() and (labels_in_manifest.to_numpy() == predictions["y_true"].to_numpy()).all(),
        f"{directory}: imágenes fuera de validation o etiquetas distintas del manifiesto.",
    )

    metrics = compute_metrics(predictions["y_true"], predictions["probability_malignant"])
    for metric, key in SAVED_KEYS.items():
        require(abs(metrics[metric] - saved_metrics[key]) < 1e-6, f"{directory}: {metric} recalculada no coincide con validation_metrics.json.")

    return {"config": config, "history": history, "predictions": predictions, "metrics": metrics}

## 2. Línea base B0

Se leen las configuraciones, historiales, métricas y predicciones existentes de R4. No se cargan los checkpoints `.pth` ni se modifica nada. Al cargar cada corrida se comprueba que el checkpoint sea la época de máximo `val_roc_auc`, que las predicciones sean las de esa época y que las métricas recalculadas desde ellas coincidan con `validation_metrics.json`.

In [ ]:
B0 = {seed: load_run(b0_dir(seed), seed) for seed in SEEDS}

pd.DataFrame(
    [
        {
            "seed": seed,
            "épocas ejecutadas": B0[seed]["config"]["epochs_run"],
            "época del checkpoint": B0[seed]["config"]["epoch_best"],
            **{metric: round(B0[seed]["metrics"][metric], 4) for metric in SAVED_KEYS},
        }
        for seed in SEEDS
    ]
)

## 3. Pesos esperados de B2 y comandos de entrenamiento

Los pesos se calculan a partir de las frecuencias de entrenamiento (800 imágenes). El orden corresponde a las etiquetas ordenadas: posición 0 = benigna, posición 1 = maligna.

La configuración de B2 se deriva del `run_config.json` de cada semilla de B0: se conservan todos los campos de configuración, incluido el early stopping, y solo cambian los pesos de clase (`class_weights`), el exponente registrado (`class_weight_alpha`) y la etiqueta del experimento. Los campos de resultado (épocas ejecutadas, época del checkpoint y valores en el checkpoint) dependen de cada corrida y no se comparan.

Cada entrenamiento invoca el mismo punto de entrada y el mismo YAML que B0 (`src.classification.run_experiment`), con `--class-weight-alpha 0.5` y una carpeta de salida propia de R7. Hay dos formas equivalentes de lanzarlos en Falcon:

- **Desde la terminal**, en la raíz del proyecto, con los comandos que imprime la primera celda de esta sección.
- **Desde este notebook**, poniendo `RUN_TRAINING = True`. La última celda de esta sección ejecuta esos mismos comandos en secuencia, con el intérprete del kernel (que debe ser el entorno con PyTorch y GPU). Una semilla cuya carpeta ya contiene archivos se omite: nunca se sobrescribe ni se usa `--overwrite`.

In [ ]:
train_labels = manifest.loc[manifest["experimental_split"] == "train", "binary_label"].to_numpy()
WEIGHTS_B0 = compute_class_weights(train_labels).tolist()
WEIGHTS_B2 = compute_class_weights(train_labels, alpha=ALPHA).tolist()

class_counts = np.bincount(train_labels)
display(
    pd.DataFrame(
        {
            "clase": ["benigna (0)", "maligna (1)"],
            "n entrenamiento": class_counts,
            "peso B0 (α = 1.0)": np.round(WEIGHTS_B0, 4),
            f"peso B2 (α = {ALPHA})": np.round(WEIGHTS_B2, 4),
        }
    )
)
print(
    f"Relación maligna/benigna: B0 = {WEIGHTS_B0[1] / WEIGHTS_B0[0]:.2f}; B2 = {WEIGHTS_B2[1] / WEIGHTS_B2[0]:.2f}. "
    f"Peso medio por muestra: B0 = {np.dot(class_counts, WEIGHTS_B0) / class_counts.sum():.4f}; B2 = {np.dot(class_counts, WEIGHTS_B2) / class_counts.sum():.4f}."
)

OUTCOME_FIELDS = [
    "epochs_run", "epoch_best", "val_weighted_loss_at_best_checkpoint", "val_unweighted_loss_at_best_checkpoint",
    "val_acc_at_best_checkpoint", "early_stopped", "early_stopped_at_epoch", "val_roc_auc_best",
]


def expected_b2_config(b0_config):
    # Configuración de B0 sin resultados ni pesos; los pesos efectivos se comparan aparte
    expected = {key: value for key, value in b0_config.items() if key not in OUTCOME_FIELDS and key != "class_weights"}
    expected.update({"experiment": EXPERIMENT_NAME, "class_weight_alpha": ALPHA})
    return expected


RUN_TRAINING = False  # True para entrenar B2 desde este notebook (Falcon); False solo muestra los comandos


def b2_command(seed, python="python"):
    return [
        python, "-m", "src.classification.run_experiment",
        "--arch", ARCH, "--condition", CONDITION,
        "--experiment", EXPERIMENT_NAME, "--seed", str(seed),
        "--checkpoint-metric", "val_roc_auc", "--class-weight-alpha", str(ALPHA),
        "--output-dir", b2_dir(seed).relative_to(PROJECT_ROOT).as_posix(),
    ]


print("\nComandos equivalentes para la terminal de Falcon (uno por semilla):\n")
for seed in SEEDS:
    print(shlex.join(b2_command(seed)), end="\n\n")

In [ ]:
if not RUN_TRAINING:
    print("RUN_TRAINING = False: no se inicia ningún entrenamiento.")
else:
    for seed in SEEDS:
        output_dir = b2_dir(seed)
        print("=" * 80)
        print(f"Semilla {seed} -> resultados en {output_dir.relative_to(PROJECT_ROOT).as_posix()}/")

        if output_dir.exists() and any(output_dir.iterdir()):
            print("Omitida: la carpeta ya contiene archivos y no se sobrescribe. Si una corrida quedó incompleta, revísela o muévala manualmente.")
            continue

        process = subprocess.Popen(
            b2_command(seed, sys.executable), cwd=PROJECT_ROOT,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace", bufsize=1,
        )
        try:
            for line in process.stdout:
                print(line, end="")
        except KeyboardInterrupt:
            process.terminate()  # al interrumpir la celda no debe quedar un entrenamiento huérfano
            raise
        if process.wait() != 0:
            raise RuntimeError(f"El entrenamiento de la semilla {seed} terminó con error (código {process.returncode}).")

## 4. Carga y verificación de B2

Esta sección solo lee archivos y requiere que las tres corridas hayan terminado; con `RUN_TRAINING = False` puede ejecutarse las veces que haga falta sin repetir entrenamientos. Además de las comprobaciones internas de cada corrida (sección 1), se verifica lo necesario para que B0 y B2 sean comparables:

- la configuración de B2 coincide con la esperada (early stopping incluido), salvo los pesos de clase, su exponente y los campos de resultado;
- los pesos efectivos registrados son los calculados en la sección 3;
- B2 se evaluó sobre las mismas imágenes de validation, en el mismo orden, que B0.

In [ ]:
B2_FILES = [
    "run_config.json", "training_history.json", "validation_metrics.json",
    "val_predictions_best_auc.csv", "best_auc_meta.json", "best_auc.pth",
]
missing = [b2_dir(seed) / name for seed in SEEDS for name in B2_FILES if not (b2_dir(seed) / name).exists()]
if missing:
    raise FileNotFoundError("Faltan salidas de B2. Complete primero los entrenamientos de la sección 3 (desde este notebook o la terminal):\n" + "\n".join(str(p) for p in missing))

B2 = {seed: load_run(b2_dir(seed), seed) for seed in SEEDS}

for seed in SEEDS:
    config = B2[seed]["config"]
    observed = {key: value for key, value in config.items() if key not in OUTCOME_FIELDS and key != "class_weights"}
    expected = expected_b2_config(B0[seed]["config"])
    differences = {
        key: (observed.get(key), expected.get(key))
        for key in set(observed) | set(expected)
        if observed.get(key) != expected.get(key)
    }
    require(not differences, f"B2 semilla {seed}: la configuración difiere de la esperada (observado, esperado): {differences}")
    require(np.allclose(config["class_weights"], WEIGHTS_B2, rtol=1e-6), f"B2 semilla {seed}: los pesos registrados {config['class_weights']} no son los esperados {WEIGHTS_B2}.")
    require(
        B2[seed]["predictions"]["image_id"].tolist() == B0[seed]["predictions"]["image_id"].tolist(),
        f"Semilla {seed}: B0 y B2 no se evaluaron sobre las mismas imágenes en el mismo orden.",
    )

print("B2 verificado: misma configuración que B0 salvo los pesos de clase, y mismas imágenes de validation.")

## 5. Comparación B0 frente a B2

Las métricas se calculan con las probabilidades del checkpoint de cada corrida; sensibilidad y especificidad usan el umbral 0.5. Las diferencias son emparejadas por semilla (B2 − B0). Con pesos distintos, las probabilidades de B2 no tienen la misma calibración que las de B0, de modo que los cambios en sensibilidad y especificidad al umbral 0.5 reflejan también ese desplazamiento; AP y ROC-AUC no dependen del umbral.

La tabla por semilla y el resumen (media y desviación estándar muestral entre las tres semillas) se exportan a `class_weights/analysis/`.

In [ ]:
METRICS = list(SAVED_KEYS)

rows = []
for seed in SEEDS:
    row = {
        "seed": seed,
        "epocas_B0": B0[seed]["config"]["epochs_run"],
        "epocas_B2": B2[seed]["config"]["epochs_run"],
        "epoca_checkpoint_B0": B0[seed]["config"]["epoch_best"],
        "epoca_checkpoint_B2": B2[seed]["config"]["epoch_best"],
    }
    for metric in METRICS:
        row[f"{metric}_B0"] = B0[seed]["metrics"][metric]
        row[f"{metric}_B2"] = B2[seed]["metrics"][metric]
        row[f"delta_{metric}"] = row[f"{metric}_B2"] - row[f"{metric}_B0"]
    rows.append(row)
per_seed = pd.DataFrame(rows)

summary = pd.DataFrame(
    [
        {
            "metrica": metric,
            "media_B0": per_seed[f"{metric}_B0"].mean(),
            "de_B0": per_seed[f"{metric}_B0"].std(ddof=1),
            "media_B2": per_seed[f"{metric}_B2"].mean(),
            "de_B2": per_seed[f"{metric}_B2"].std(ddof=1),
            "delta_medio": per_seed[f"delta_{metric}"].mean(),
            "de_delta": per_seed[f"delta_{metric}"].std(ddof=1),
        }
        for metric in METRICS
    ]
)

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
per_seed.to_csv(ANALYSIS_DIR / "b0_vs_b2_por_semilla.csv", index=False)
summary.to_csv(ANALYSIS_DIR / "b0_vs_b2_resumen.csv", index=False)

display(per_seed.round(4))
display(summary.round(4))

### 5.1 Evolución del entrenamiento

Pérdidas y métricas de validation por época. Las líneas punteadas marcan el fin del entrenamiento de cada configuración por early stopping y las estrellas indican su checkpoint. Las pérdidas ponderadas (`train_loss` y `val_loss`, marcadas con asterisco) usan pesos distintos en B0 y B2, por lo que sus escalas no son comparables; `val_unweighted_loss`, `val_average_precision` y `val_roc_auc` sí lo son.

In [ ]:
SERIES = [
    ("train_loss", "train_loss (ponderada)*"),
    ("val_loss", "val_loss (ponderada)*"),
    ("val_unweighted_loss", "val_unweighted_loss"),
    ("val_average_precision", "val_average_precision"),
    ("val_roc_auc", "val_roc_auc"),
]

fig, axes = plt.subplots(len(SERIES), len(SEEDS), figsize=(15, 14), sharex=True, sharey="row")
for col, seed in enumerate(SEEDS):
    for row, (key, label) in enumerate(SERIES):
        ax = axes[row, col]
        for tag, run in (("B0", B0[seed]), ("B2", B2[seed])):
            history, best = run["history"], run["config"]["epoch_best"]
            ax.plot(history.index, history[key], color=COLORS[tag], marker="o", ms=3, label=LABELS[tag] if row == col == 0 else None)
            ax.scatter([best], [history.loc[best, key]], marker="*", s=150, color=COLORS[tag], edgecolors="black", zorder=5)
            ax.axvline(run["config"]["epochs_run"], color=COLORS[tag], ls=":", lw=1.4)
        if row == 0:
            ax.set_title(f"Semilla {seed}")
        if col == 0:
            ax.set_ylabel(label)
        if row == len(SERIES) - 1:
            ax.set_xlabel("Época")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Figura 1. Evolución del entrenamiento: B0 (α = 1.0) y B2 (α = 0.5). *Pérdidas ponderadas con pesos distintos en cada configuración", y=1.0)
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / "b0_vs_b2_curvas_entrenamiento.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2 Average Precision por semilla

AP de cada semilla en B0 y B2, con la diferencia emparejada anotada.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for _, row in per_seed.iterrows():
    ax.plot([0, 1], [row["average_precision_B0"], row["average_precision_B2"]], color="0.6", lw=1.2, zorder=1)
    ax.scatter([0], [row["average_precision_B0"]], color=COLORS["B0"], s=55, zorder=3)
    ax.scatter([1], [row["average_precision_B2"]], color=COLORS["B2"], s=55, zorder=3)
    ax.annotate(f"seed {int(row['seed'])}  (Δ = {row['delta_average_precision']:+.3f})", (1.04, row["average_precision_B2"]), fontsize=8, va="center")

ax.set_xticks([0, 1])
ax.set_xticklabels([LABELS["B0"], LABELS["B2"]])
ax.set_xlim(-0.3, 1.9)
ax.set_ylabel("Average Precision (validation)")
ax.set_title("Figura 2. AP de B0 y B2 por semilla")
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / "b0_vs_b2_ap_por_semilla.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.3 Pérdida sin ponderar en el checkpoint

Complemento descriptivo, solo dentro del notebook. La pérdida de validation sin ponderar es la única medida de pérdida comparable entre B0 y B2.

In [ ]:
pd.DataFrame(
    [
        {
            "seed": seed,
            "val_unweighted_loss en el checkpoint, B0": B0[seed]["history"].loc[B0[seed]["config"]["epoch_best"], "val_unweighted_loss"],
            "val_unweighted_loss en el checkpoint, B2": B2[seed]["history"].loc[B2[seed]["config"]["epoch_best"], "val_unweighted_loss"],
            "val_unweighted_loss mínima, B0": B0[seed]["history"]["val_unweighted_loss"].min(),
            "val_unweighted_loss mínima, B2": B2[seed]["history"]["val_unweighted_loss"].min(),
        }
        for seed in SEEDS
    ]
).round(4)

## 6. Limitaciones

- Son tres semillas y 22 imágenes malignas en validation: cada falso negativo equivale a 4.5 puntos de sensibilidad. Las diferencias emparejadas describen lo observado y no constituyen una prueba de efecto.
- Los pesos de clase intervienen también en el criterio de early stopping (pérdida ponderada de validation). Una diferencia entre B0 y B2 puede deberse a los pesos, a un cambio en la época de parada o a ambos; este experimento no los separa.
- Con pesos distintos cambia la calibración de las probabilidades, por lo que sensibilidad y especificidad al umbral 0.5 no son estrictamente comparables entre configuraciones. No se optimiza ningún umbral.
- Solo se usa validation; los resultados no estiman el desempeño en datos no vistos.
- El entorno de B0 no quedó registrado y el entrenamiento no fuerza el modo determinista de cuDNN: parte de la diferencia emparejada puede ser variabilidad de ejecución.
- Solo se guardan las probabilidades del checkpoint seleccionado, y no hay identificadores de paciente para verificar independencia entre imágenes.
- Este experimento no explica por qué cambian las métricas; esa interpretación corresponde a análisis posteriores.